# Data Acquisition

In this lab, we will explore the process of data acquisition. 

In the case of *passive* network traffic analysis, there are generally two primary ways of acquiring data:
* Packet capture
* Network traffic flows (sometimes called IPFIX)

The advent of more programmability in networks is quickly changing this landscape. 

In particular, systems like Retina are now making it possible to ask more complex questions of network traffic from passive traffic capture and analysis but the general underlying traffic patterns are still based on raw packet capture.

## Background

Because packet captures are so large, it can sometimes be convenient to work with summary statistics about network traffic. Instead of the raw packets, data could represent the total number of bytes, packets, and so forth for flows. 

Raw traffic capture is thus sometimes represented as summaries of flow statistics, rather than raw packet traces. In this activity, we will *generate* the summary statistics and then think about what types of information is (and is not) available in a packet trace summary vs. a raw packet capture.

## Step 1: Load a Packet Trace

Load the packet capture from the last assignment.

In [ ]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
5,6,2018-02-11 08:10:00.547036,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,96,Standard query 0xb756 A r4---sn-gxo5uxg-jqbe.g...
6,7,2018-02-11 08:10:00.547156,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x62ab A ssl.gstatic.com
7,8,2018-02-11 08:10:00.547249,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,74,Standard query 0x42fb A www.google.com
8,9,2018-02-11 08:10:00.853950,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,386,Standard query response 0x11d3 A 216.58.213.162
9,10,2018-02-11 08:10:00.853970,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x8756 A www.gstatic.com


In [2]:
ndf.shape[0] #number of packets

141471

## Step 2: Generate Statistics for Each Flow

A **flow** is defined as groups of packets that share the following attributes:
* Source IP Address
* Destination IP Address
* Source Port
* Destination Port
* Time interval

The csv file we used in the past assignments do not have port numbers, so you can simply group on source and destination IP address.

For each flow in the packet trace, generate the following statistics for each flow:

* Number of bytes
* Number of packets
* Duration (time)

In [9]:
flows_ndf = ndf.groupby(["Source", "Destination", "Protocol"]).count()
flows_ndf

No.  Time  Length  \
Source                   Destination           Protocol                      
0.0.0.0                  255.255.255.255       BOOTP       8     8       8   
                         all-systems.mcast.net IGMPv2      4     4       4   
104.31.113.215           192.168.43.72         HTTP        2     2       2   
                                               TCP         4     4       4   
17.188.166.20            192.168.43.72         TCP         2     2       2   
...                                                      ...   ...     ...   
par10s38-in-f3.1e100.net 192.168.43.72         TCP        63    63      63   
                                               TLSv1.2    47    47      47   
par21s03-in-f2.1e100.net 192.168.43.72         SSL         1     1       1   
                                               TCP         7     7       7   
                                               TLSv1.2     7     7       7   

                                                         Info  
Source                   Destination           Protocol        
0.0.0.0                  255.255.255.255       BOOTP        8  
                         all-systems.mcast.net IGMPv2       4  
104.31.113.215           192.168.43.72         HTTP         2  
                                               TCP          4  
17.188.166.20            192.168.43.72         TCP          2  
...                                                       ...  
par10s38-in-f3.1e100.net 192.168.43.72         TCP         63  
                                               TLSv1.2     47  
par21s03-in-f2.1e100.net 192.168.43.72         SSL          1  
                                               TCP          7  
                                               TLSv1.2      7  

[148 rows x 4 columns]

### Total Number of Flows

Count the total number of flows in this trace.

148 flows

### Number of Bytes

Count the total number of bytes for each flow in the trace. 

Then, sort the flows by size, in bytes.  

What do you notice about the large flows? What do they look like?

In [15]:
#Sorted by # of bytes
#.sort_values(ascending = False) #most larger flows look like video streaming flows, mostly Netflix?

flow_no = ndf.groupby(["Source", "Destination", "Protocol"])["No."].count()
flow_no.sort_values(ascending = False)

Source                                   Destination                              Protocol
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSL         77678
192.168.43.72                            ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  TCP         47744
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSL          4605
192.168.43.72                            ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  TCP          3111
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            TCP          2207
                                                                                              ...  
Netgear_bb:19:ee                         Apple_01:4c:54                           EAPOL           1
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSLv2           1
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSLv2           1
par10s29-

In [16]:
#Sorted by size
flow_size = ndf.groupby(["Source", "Destination", "Protocol"])["Length"].count()
flow_size.sort_values(ascending = False)

Source                                   Destination                              Protocol
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSL         77678
192.168.43.72                            ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  TCP         47744
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSL          4605
192.168.43.72                            ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  TCP          3111
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            TCP          2207
                                                                                              ...  
Netgear_bb:19:ee                         Apple_01:4c:54                           EAPOL           1
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSLv2           1
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSLv2           1
par10s29-

### Number of Packets

Count the number of packets in each flow. 

What do you notice about these flows? Are they similar to the largest flows by bytes? Which differ?

In [ ]:
# of packets in each flow
import matplotlib.pyplot as plt

plt.bar(flows_ndf[("Source", "Destination", "Protocol") ], flows_ndf["No."])
plt.show()

No.  Time  Length  \
Source                   Destination           Protocol                      
0.0.0.0                  255.255.255.255       BOOTP       8     8       8   
                         all-systems.mcast.net IGMPv2      4     4       4   
104.31.113.215           192.168.43.72         HTTP        2     2       2   
                                               TCP         4     4       4   
17.188.166.20            192.168.43.72         TCP         2     2       2   
...                                                      ...   ...     ...   
par10s38-in-f3.1e100.net 192.168.43.72         TCP        63    63      63   
                                               TLSv1.2    47    47      47   
par21s03-in-f2.1e100.net 192.168.43.72         SSL         1     1       1   
                                               TCP         7     7       7   
                                               TLSv1.2     7     7       7   

                                                         Info  
Source                   Destination           Protocol        
0.0.0.0                  255.255.255.255       BOOTP        8  
                         all-systems.mcast.net IGMPv2       4  
104.31.113.215           192.168.43.72         HTTP         2  
                                               TCP          4  
17.188.166.20            192.168.43.72         TCP          2  
...                                                       ...  
par10s38-in-f3.1e100.net 192.168.43.72         TCP         63  
                                               TLSv1.2     47  
par21s03-in-f2.1e100.net 192.168.43.72         SSL          1  
                                               TCP          7  
                                               TLSv1.2      7  

[148 rows x 4 columns]

A few of them differ

### Duration

Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

What are the longest flows in the trace?

In [6]:
flows_ndf["Time"].sort_values(ascending = False)

Source                                   Destination                              Protocol
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSL         77678
192.168.43.72                            ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  TCP         47744
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSL          4605
192.168.43.72                            ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  TCP          3111
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            TCP          2207
                                                                                              ...  
Netgear_bb:19:ee                         Apple_01:4c:54                           EAPOL           1
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSLv2           1
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net  192.168.43.72                            SSLv2           1
par10s29-

## Bytes and Packets Per Second

Compute the bytes per second and packets per second for each flow.

For a simple feature computation, compute the average bytes and packets per second for each flow, for the entire duration of the flow.  If you want to get more clever or fancy, you can do "windowed averages", computing bytes or packets per second for shorter time intervals.

## Note

Some of the libraries that we will use in this class, including the `netml` library from the University of Chicago, will compute these and other statistics automatically.

## Thought Questions

1. What are the largest flows in terms of: Number of bytes? Number of packets?

2. What do you notice about the flow sizes and the directions of flows?

3. What kinds of features are *not* available in packet summary statistics like those above which might be available in a raw packet trace? How might those features be useful for different packet classification problems?